In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict


# =========================================================
# 1. STATE
# =========================================================

class State(TypedDict):
    count: int


# =========================================================
# 2. NODES
# =========================================================

def add_one(state: State):
    print("ADD_ONE running...")

    return {
        "count": state["count"] + 1
    }


def add_two(state: State):
    print("ADD_TWO running...")

    return {
        "count": state["count"] + 2
    }


def add_three(state: State):
    print("ADD_THREE running...")

    return {
        "count": state["count"] + 3
    }


# =========================================================
# 3. BUILD GRAPH
# =========================================================

builder = StateGraph(State)

builder.add_node("add_one", add_one)
builder.add_node("add_two", add_two)
builder.add_node("add_three", add_three)

builder.add_edge(START, "add_one")
builder.add_edge("add_one", "add_two")
builder.add_edge("add_two", "add_three")
builder.add_edge("add_three", END)


# =========================================================
# 4. CHECKPOINTER
# =========================================================

checkpointer = InMemorySaver()

graph = builder.compile(
    checkpointer=checkpointer
)


# =========================================================
# 5. ORIGINAL THREAD
# =========================================================

original_config = {
    "configurable": {
        "thread_id": "thread-001"
    }
}


print("\n========================================")
print("1. ORIGINAL EXECUTION")
print("========================================")

original_result = graph.invoke(
    {"count": 0},
    config=original_config
)

print(
    "\nOriginal Final Result:",
    original_result
)


# =========================================================
# 6. GET CHECKPOINT HISTORY
# =========================================================

history = list(
    graph.get_state_history(
        original_config
    )
)


print("\n========================================")
print("2. CHECKPOINT HISTORY")
print("========================================")

for index, state in enumerate(history):

    checkpoint_id = (
        state.config[
            "configurable"
        ]["checkpoint_id"]
    )

    print(f"\nCheckpoint {index}")
    print("Checkpoint ID:", checkpoint_id)
    print("Values:", state.values)
    print("Next:", state.next)


# =========================================================
# 7. SELECT OLD CHECKPOINT
# =========================================================

target_state = None

for state in history:

    # We need a checkpoint from which
    # execution can continue.

    if state.next:

        target_state = state
        break


if target_state is None:

    print(
        "\nNo suitable checkpoint found."
    )

else:

    old_checkpoint_id = (
        target_state.config[
            "configurable"
        ]["checkpoint_id"]
    )


    # =====================================================
    # 8. OLD CHECKPOINT CONFIG
    # =====================================================

    old_checkpoint_config = {
        "configurable": {
            "thread_id": "thread-001",
            "checkpoint_id": old_checkpoint_id
        }
    }


    # =====================================================
    # 9. GET OLD STATE
    # =====================================================

    old_state = graph.get_state(
        old_checkpoint_config
    )


    print("\n========================================")
    print("3. OLD CHECKPOINT")
    print("========================================")

    print(
        "Thread ID:",
        old_state.config[
            "configurable"
        ]["thread_id"]
    )

    print(
        "Checkpoint ID:",
        old_checkpoint_id
    )

    print(
        "Old Values:",
        old_state.values
    )

    print(
        "Next Node:",
        old_state.next
    )


    # =====================================================
    # 10. CREATE FORKED THREAD
    # =====================================================

    fork_config = {
        "configurable": {
            "thread_id": "thread-002"
        }
    }


    # =====================================================
    # 11. USE OLD STATE AS FORK STARTING STATE
    # =====================================================

    fork_initial_state = {
        "count": old_state.values["count"]
    }


    print("\n========================================")
    print("4. FORKED THREAD")
    print("========================================")

    print(
        "Fork Thread ID:",
        fork_config[
            "configurable"
        ]["thread_id"]
    )

    print(
        "Fork Starting State:",
        fork_initial_state
    )


    # =====================================================
    # 12. INITIAL FORK EXECUTION
    # =====================================================

    fork_result = graph.invoke(
        fork_initial_state,
        config=fork_config
    )


    print("\nFork Initial Result:")
    print(fork_result)


    # =====================================================
    # 13. GET FORK HISTORY
    # =====================================================

    fork_history = list(
        graph.get_state_history(
            fork_config
        )
    )


    # =====================================================
    # 14. SELECT FORK CHECKPOINT
    # =====================================================

    fork_target = None

    for state in fork_history:

        if state.next:

            fork_target = state
            break


    if fork_target is None:

        print(
            "\nNo fork checkpoint found."
        )

    else:

        fork_checkpoint_id = (
            fork_target.config[
                "configurable"
            ]["checkpoint_id"]
        )


        fork_checkpoint_config = {
            "configurable": {
                "thread_id": "thread-002",
                "checkpoint_id": fork_checkpoint_id
            }
        }


        # =================================================
        # 15. MODIFY FORKED STATE
        # =================================================

        print("\n========================================")
        print("5. MODIFY FORKED STATE")
        print("========================================")

        print(
            "Before Update:",
            fork_target.values
        )


        graph.update_state(
            fork_checkpoint_config,
            {
                "count": 10
            }
        )


        modified_state = graph.get_state(
            fork_checkpoint_config
        )


        print(
            "After Update:",
            modified_state.values
        )


        # =================================================
        # 16. CONTINUE FORKED EXECUTION
        # =================================================

        print("\n========================================")
        print("6. CONTINUE FORKED EXECUTION")
        print("========================================")

        modified_result = graph.invoke(
            None,
            config=fork_checkpoint_config
        )


        print(
            "\nModified Fork Result:",
            modified_result
        )


    # =====================================================
    # 17. GET ORIGINAL FINAL STATE
    # =====================================================

    original_final_state = graph.get_state(
        original_config
    )


    # =====================================================
    # 18. GET FORK FINAL STATE
    # =====================================================

    fork_final_state = graph.get_state(
        fork_config
    )


    # =====================================================
    # 19. FINAL COMPARISON
    # =====================================================

    print("\n========================================")
    print("7. FINAL COMPARISON")
    print("========================================")

    print("\nORIGINAL THREAD")
    print(
        "Thread ID:",
        original_config[
            "configurable"
        ]["thread_id"]
    )

    print(
        "Final State:",
        original_final_state.values
    )

    print(
        "Checkpoint:",
        original_final_state.config[
            "configurable"
        ]["checkpoint_id"]
    )


    print("\nFORKED THREAD")
    print(
        "Thread ID:",
        fork_config[
            "configurable"
        ]["thread_id"]
    )

    print(
        "Final State:",
        fork_final_state.values
    )

    print(
        "Checkpoint:",
        fork_final_state.config[
            "configurable"
        ]["checkpoint_id"]
    )


    # =====================================================
    # 20. CONCEPT SUMMARY
    # =====================================================

    print("\n========================================")
    print("CONCEPT SUMMARY")
    print("========================================")

    print("""
thread_id
    ↓
identifies an execution history

checkpoint_id
    ↓
identifies a specific saved state

get_state()
    ↓
reads a specific state

get_state_history()
    ↓
shows the execution timeline

update_state()
    ↓
modifies a selected checkpoint/state

fork
    ↓
creates an alternate execution path
""")